# Future Interns - Data Science & Analytics
## Task 3: Marketing Funnel & Conversion Performance Analysis
**Author:** Data Science Intern  
**Repository:** `FUTURE_DS_03`  
**Objective:** Analyze omnichannel marketing funnel data to identify conversion drop-offs, evaluate channel CAC and ROAS, pinpoint sales bottlenecks, and recommend data-driven growth strategies.

### 1. Import Dependencies & Setup Aesthetics

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Marketing analytics libraries loaded successfully!")

### 2. Ingest & Inspect Cleaned Marketing Funnel Dataset

In [2]:
data_path = os.path.join("..", "data", "cleaned_marketing_funnel_data.csv")
df = pd.read_csv(data_path)
print(f"Audited Touchpoints: {df.shape[0]:,} records across {df.shape[1]} fields")
display(df.head(3))

### 3. Executive Marketing Funnel Scorecard

In [3]:
visitors = len(df)
leads = df["Lead_Captured"].sum()
mqls = df["MQL"].sum()
sqls = df["SQL"].sum()
opps = df["Opportunity"].sum()
won = df["Closed_Won"].sum()
spend = df["Ad_Spend"].sum()
revenue = df["Revenue"].sum()

print("=" * 50)
print("     MARKETING FUNNEL EXECUTIVE SCORECARD")
print("=" * 50)
print(f"Inbound Web Visitors    : {visitors:,}")
print(f"Leads Captured          : {leads:,} ({leads/visitors*100:.2f}%)")
print(f"Marketing Qualified     : {mqls:,} ({mqls/visitors*100:.2f}%)")
print(f"Sales Qualified         : {sqls:,} ({sqls/visitors*100:.2f}%)")
print(f"Opportunities (Demos)   : {opps:,} ({opps/visitors*100:.2f}%)")
print(f"Closed Won Customers    : {won:,} ({won/visitors*100:.2f}%)")
print(f"Total Ad Spend          : ${spend:,.2f}")
print(f"Attributed Pipeline Rev : ${revenue:,.2f}")
print(f"Blended ROAS            : {revenue/spend:.2f}x")
print(f"Blended CAC             : ${spend/won:,.2f} per customer")
print("=" * 50)

### 4. Stage Progression & Conversion Drop-Off Analysis

In [4]:
stages = ['Visitors', 'Leads', 'MQL', 'SQL', 'Opportunity', 'Closed Won']
counts = [visitors, leads, mqls, sqls, opps, won]
funnel_summary = pd.DataFrame({'Stage': stages, 'Volume': counts})
funnel_summary['Share_of_Total_%'] = (funnel_summary['Volume'] / visitors) * 100
display(funnel_summary)

plt.figure(figsize=(10, 5))
plt.barh(stages[::-1], counts[::-1], color="#3B82F6")
plt.title("Prospect Funnel Volume by Milestone", fontsize=14, fontweight="bold")
plt.xlabel("Count of Prospects")
plt.show()

### 5. Multi-Channel Efficiency (ROAS & CAC Matrix)

In [5]:
channel_perf = df.groupby("Channel").agg({
    "Lead_ID": "count",
    "Closed_Won": "sum",
    "Ad_Spend": "sum",
    "Revenue": "sum"
}).reset_index()
channel_perf.rename(columns={"Lead_ID": "Visitors", "Closed_Won": "Customers"}, inplace=True)
channel_perf["Conv_Rate_%"] = (channel_perf["Customers"] / channel_perf["Visitors"]) * 100
channel_perf["CAC"] = channel_perf["Ad_Spend"] / channel_perf["Customers"]
channel_perf["ROAS"] = channel_perf["Revenue"] / channel_perf["Ad_Spend"]
display(channel_perf.sort_values("ROAS", ascending=False))

plt.figure(figsize=(10, 5))
sns.barplot(data=channel_perf.sort_values("ROAS", ascending=False), x="Channel", y="ROAS", palette="Greens_r")
plt.xticks(rotation=30, ha="right")
plt.axhline(3.0, color="red", linestyle="--", label="Target ROAS (3.0x)")
plt.title("Return on Ad Spend (ROAS) by Channel", fontsize=14, fontweight="bold")
plt.legend()
plt.show()

### 6. Sales Opportunity Loss Reasons

In [6]:
lost = df[(df["Opportunity"] == 1) & (df["Closed_Won"] == 0)]["Lost_Reason"].value_counts()
display(lost)

lost.plot(kind="barh", color="#EF4444", figsize=(10, 4))
plt.title("Root Cause Analysis for Lost Opportunities", fontsize=14, fontweight="bold")
plt.xlabel("Count")
plt.show()

### 7. Strategic Recommendations
1. **Budget Reallocation:** Pull back on Paid Social (1.24x ROAS) and fund high-performing Referral/SEO channels (6.8x+ ROAS).
2. **Optimize Mobile Experience:** Mobile converts at less than half the rate of desktop due to high friction.
3. **Speed to Lead:** Accelerate SDR response time to improve MQL to SQL qualification.